# Notebook 03: Model training (E0 pilot and E1)

**Framework component:** F3, model training and selection. Needs the frozen manifest from Notebook 02 (protocol v2).

This notebook:

1. checks the frozen manifest against its recorded SHA-256, so training uses exactly the protocol v2 split;
2. **E0 pilot:** for each architecture, times a few training steps, and converts a model to LiteRT (FP32 and INT8) to confirm that training and conversion work together;
3. **E1:** trains MobileNetV2, MobileNetV3-Small and EfficientNet-B0 on the cassava (3 classes) and maize (5 classes) tasks with seeds 17, 42 and 2026, following Training recipe: 18 runs;
4. saves each run's best model, training history and **selection-set** predictions to Google Drive.

The internal test and the Ghanaian images are **not loaded** here. They are used only after every choice is frozen.

**Runtime:** GPU (Runtime > Change runtime type > **A100 GPU**, or L4; T4 also works but is slowest). To finish sooner, open this notebook in **two sessions** and set `CROPS_TO_RUN = ['cassava']` in one and `['maize']` in the other (first code cell). Each run takes roughly 10 to 30 minutes, so all 18 need several hours. **You can stop at any time.** Every finished run is saved, and running all cells again continues from the next unfinished run. If Colab's free GPU time runs out, continue the next day.

In [ ]:
# 1. Setup, GPU check and environment record
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, math
import numpy as np, pandas as pd
import tensorflow as tf
import keras
from sklearn.metrics import f1_score, accuracy_score

# Which crops this session trains. To use two Colab sessions at once, set ['cassava'] in one and ['maize'] in the other.
CROPS_TO_RUN = ['cassava', 'maize']

DRIVE = '/content/drive/MyDrive/ocdf'
WORK = '/content/working'
PROTOCOL_VERSION = 'v2'
OUT = f'{DRIVE}/records/nb03'; RUNS = f'{DRIVE}/runs'
for d in [OUT, RUNS, WORK]: os.makedirs(d, exist_ok=True)

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise SystemExit('No GPU found. Choose Runtime > Change runtime type > T4 GPU, then run all cells again.')
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
try: gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except Exception: gpu_name = str(gpus)
env = {'notebook': '03_training', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'gpu': gpu_name,
       'cpu_count': os.cpu_count(), 'platform': platform.platform()}
try:
    import sklearn; env['scikit_learn'] = sklearn.__version__
except Exception: pass
json.dump(env, open(f'{OUT}/environment_nb03.json', 'w'), indent=1)
print(json.dumps(env, indent=1))

In [ ]:
# 2. Frozen manifest (integrity check) and working images for D1, D4, D6 only
protocol = json.load(open(f'{DRIVE}/records/protocol_{PROTOCOL_VERSION}.json'))
mpath = f'{DRIVE}/manifests/{protocol["manifest"]}'
sha = hashlib.sha256(open(mpath, 'rb').read()).hexdigest()
assert sha == protocol['manifest_sha256'], f'Manifest changed since it was frozen: {sha}'
print('manifest', protocol['manifest'], 'SHA-256 matches the frozen protocol:', sha[:16], '...')
M = pd.read_csv(mpath, dtype=str)
DEV = M[M.partition.isin(['train', 'select'])].copy()          # test, calibration, threshold and external rows are not used here
print(DEV.groupby(['crop', 'partition', 'research_label']).size().unstack(fill_value=0))
reg_path = f'{DRIVE}/records/release_register_v1.json'
if os.path.exists(reg_path):                                   # confirms the image archives are intact (e.g. after copying between accounts)
    reg = json.load(open(reg_path))
    for src in ['D1', 'D4', 'D6']:
        tp = f'{DRIVE}/working/working_{src}.tar'
        h = hashlib.sha256()
        with open(tp, 'rb') as f:
            for blk in iter(lambda: f.read(1 << 22), b''): h.update(blk)
        ok = h.hexdigest() == reg[src]['working']['sha256']
        print(f'working_{src}.tar SHA-256 matches Notebook 01 record:', ok)
        assert ok, f'working_{src}.tar differs from the file Notebook 01 created; copy it again'
for src in ['D1', 'D4', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        t0 = time.time()
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
        print(f'extracted {src} in {time.time()-t0:.0f}s')
missing = [p for p in DEV.working_path if not os.path.exists(f'{WORK}/{p}')]
assert not missing, f'{len(missing)} working images missing, e.g. {missing[:3]}'

In [ ]:
# 3. Recipe (training recipe), data pipeline and model builder
IMG = 224
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M02', 'M03', 'M04', 'M05']}
ARCHS = ['MobileNetV2', 'MobileNetV3Small', 'EfficientNetB0']
SEEDS = [17, 42, 2026]
RECIPE = {'input': '224x224 RGB, aspect-preserving resize with zero padding, float 0-255 into the model',
          'head': 'GlobalAveragePooling2D, Dropout 0.2, Dense softmax', 'head_lr': 1e-3, 'head_max_epochs': 5,
          'finetune_lr': 1e-5, 'finetune_max_epochs': 25, 'loss': 'categorical cross-entropy, no label smoothing',
          'class_weights': protocol['class_weights_from_training_counts'], 'class_weight_method': 'per-image sample weight from the training-count weights', 'batch_size': 32, 'patience': 5,
          'stopping_metric': 'selection macro-F1, best weights restored',
          'augmentation': 'training only: horizontal and vertical flips, rotation up to 0.05 turn (18 degrees), brightness and contrast +/-10%',
          'unfreeze_from': {'MobileNetV2': 'block_13', 'MobileNetV3Small': 'expanded_conv_8', 'EfficientNetB0': 'block6a'},
          'batch_norm': 'backbone called with training=False (inference-mode statistics) in both phases'}
BATCH = RECIPE['batch_size']

def load_img(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG, IMG, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)   # cached as uint8 (4x less memory)

AUG = {}
def augmenter(seed):
    if seed not in AUG:
        AUG[seed] = keras.Sequential([keras.layers.RandomFlip('horizontal_and_vertical', seed=seed),
                                      keras.layers.RandomRotation(0.05, fill_mode='reflect', seed=seed),
                                      keras.layers.RandomBrightness(0.1, value_range=(0, 255), seed=seed),
                                      keras.layers.RandomContrast(0.1, seed=seed)], name=f'augment_s{seed}')
    return AUG[seed]

def base_ds(df, classes):
    # decoded, resized images cached in memory once per crop and partition (uint8, label index)
    paths = [f'{WORK}/{p}' for p in df.working_path]
    y = [classes.index(l) for l in df.research_label]
    return tf.data.Dataset.from_tensor_slices((paths, y)).map(lambda p, l: (load_img(p), l),
                                                              num_parallel_calls=tf.data.AUTOTUNE).cache()

def train_ds(base, n_classes, seed, weights):
    # training stream: shuffle, (image float 0-255, one-hot, class weight), batch, augment
    w = tf.constant(weights, tf.float32); aug = augmenter(seed)
    ds = base.shuffle(4096, seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes), tf.gather(w, l)), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).map(lambda x, y, sw: (tf.clip_by_value(aug(x, training=True), 0, 255), y, sw), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def eval_ds(base, n_classes):
    return base.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes))).batch(BATCH).prefetch(tf.data.AUTOTUNE)

def build_model(arch, n_classes):
    inp = keras.Input((IMG, IMG, 3), name='image_0_255')
    if arch == 'MobileNetV2':
        x = keras.layers.Rescaling(1 / 127.5, offset=-1, name='mnv2_scale')(inp)      # MobileNetV2 expects -1 to 1
        base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
    elif arch == 'MobileNetV3Small':
        x = inp; base = keras.applications.MobileNetV3Small(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3), include_preprocessing=True)
    elif arch == 'EfficientNetB0':
        x = inp; base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
    else: raise ValueError(arch)
    base.trainable = False
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = keras.layers.Dropout(0.2, name='dropout')(x)
    out = keras.layers.Dense(n_classes, activation='softmax', name='probs')(x)
    return keras.Model(inp, out, name=f'{arch}_{n_classes}cls'), base

def unfreeze_final_blocks(base, arch):
    prefix = RECIPE['unfreeze_from'][arch]
    names = [l.name for l in base.layers]
    idx = next((i for i, n in enumerate(names) if n.startswith(prefix)), None)
    rule = f'from first layer named {prefix}*'
    if idx is None:                                                       # layer names differ in this Keras version
        idx = int(len(names) * 0.75); rule = f'prefix {prefix} not found; final 25% of layers (from {names[idx]})'
    base.trainable = True
    for l in base.layers[:idx]: l.trainable = False
    return {'rule': rule, 'first_trainable_layer': names[idx], 'trainable_backbone_layers': len(base.layers) - idx,
            'total_backbone_layers': len(names)}

class MacroF1Stopper(keras.callbacks.Callback):
    # Selection macro-F1 each epoch; keeps the best weights; stops after `patience` epochs without improvement
    def __init__(self, sel_ds, y_sel, patience, log, phase):
        super().__init__(); self.sel_ds, self.y_sel, self.patience, self.log, self.phase = sel_ds, y_sel, patience, log, phase
        self.best, self.best_w, self.wait, self.t0 = -1.0, None, 0, None
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.sel_ds, verbose=0).argmax(1)
        f1 = f1_score(self.y_sel, p, average='macro'); acc = accuracy_score(self.y_sel, p)
        self.log.append({'phase': self.phase, 'epoch': epoch + 1, 'train_loss': float(logs.get('loss', np.nan)),
                         'train_accuracy': float(logs.get('accuracy', np.nan)), 'select_macro_f1': float(f1),
                         'select_accuracy': float(acc), 'seconds': round(time.time() - self.t0, 1)})
        print(f'  {self.phase} epoch {epoch+1}: loss {logs.get("loss", 0):.4f}  selection macro-F1 {f1:.4f}  acc {acc:.4f}  ({self.log[-1]["seconds"]}s)')
        if f1 > self.best + 1e-6: self.best, self.best_w, self.wait = f1, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience: self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None: self.model.set_weights(self.best_w)

In [ ]:
# 4. E0 pilot: time per step for each architecture, and a LiteRT conversion check (FP32 and INT8)
PILOT = f'{OUT}/e0_pilot.json'
def convert_check(model, rep_images, tag):
    res = {}
    exp_dir = f'/content/e0_export_{tag}'
    try:
        model.export(exp_dir, verbose=False); conv = lambda: tf.lite.TFLiteConverter.from_saved_model(exp_dir); route = 'Keras export + from_saved_model'
    except Exception as e:
        conv = lambda: tf.lite.TFLiteConverter.from_keras_model(model); route = f'from_keras_model (export failed: {type(e).__name__})'
    res['route'] = route
    for mode in ['fp32', 'int8']:
        try:
            c = conv()
            if mode == 'int8':
                c.optimizations = [tf.lite.Optimize.DEFAULT]
                c.representative_dataset = lambda: ([im[None, ...]] for im in rep_images)
                c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
                c.inference_input_type = tf.float32; c.inference_output_type = tf.float32
            b = c.convert()
            try:
                from ai_edge_litert.interpreter import Interpreter
            except Exception:
                Interpreter = tf.lite.Interpreter
            it = Interpreter(model_content=b); it.allocate_tensors()
            i, o = it.get_input_details()[0], it.get_output_details()[0]
            it.set_tensor(i['index'], rep_images[0][None, ...].astype(np.float32)); it.invoke()
            out = it.get_tensor(o['index'])
            res[mode] = {'ok': True, 'bytes': len(b), 'output_shape': list(out.shape), 'output_sum': float(out.sum())}
        except Exception as e:
            res[mode] = {'ok': False, 'error': f'{type(e).__name__}: {str(e)[:300]}'}
    return res

if not os.path.exists(PILOT):
    pilot = {'environment': env, 'recipe': RECIPE, 'architectures': {}}
    df = DEV[(DEV.crop == 'cassava') & (DEV.partition == 'train')].sample(640, random_state=17)
    ds = train_ds(base_ds(df, CLASSES['cassava']), 3, 17, [1.0, 1.0, 1.0])
    rep = [load_img(f'{WORK}/{p}').numpy().astype(np.float32) for p in df.working_path[:20]]
    for arch in ARCHS:
        keras.backend.clear_session(); keras.utils.set_random_seed(17)
        model, base = build_model(arch, 3)
        model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
        model.fit(ds.take(3), epochs=1, verbose=0)                               # warm-up (graph build, cache)
        t0 = time.time(); model.fit(ds, epochs=1, verbose=0); head_step = (time.time() - t0) / (640 / BATCH)
        info = unfreeze_final_blocks(base, arch)
        model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
        model.fit(ds.take(3), epochs=1, verbose=0)
        t0 = time.time(); model.fit(ds, epochs=1, verbose=0); ft_step = (time.time() - t0) / (640 / BATCH)
        conv = convert_check(model, rep, arch)
        pilot['architectures'][arch] = {'params_total': int(model.count_params()), 'unfreeze': info,
                                        'seconds_per_step_head': round(head_step, 4), 'seconds_per_step_finetune': round(ft_step, 4),
                                        'conversion': conv}
        print(arch, json.dumps(pilot['architectures'][arch], indent=1))
        del model, base; gc.collect()
    # time estimate for E1: steps per epoch x epochs x runs (upper bound, no early stopping)
    steps = {c: math.ceil(len(DEV[(DEV.crop == c) & (DEV.partition == 'train')]) / BATCH) for c in CLASSES}
    est = sum(steps[c] * (5 * a['seconds_per_step_head'] + 25 * a['seconds_per_step_finetune']) * len(SEEDS)
              for c in CLASSES for a in pilot['architectures'].values())
    pilot['e1_upper_bound_hours'] = round(est / 3600, 1)
    pilot['conversion_all_ok'] = all(a['conversion'].get(m, {}).get('ok') for a in pilot['architectures'].values() for m in ['fp32', 'int8'])
    json.dump(pilot, open(PILOT, 'w'), indent=1)
pilot = json.load(open(PILOT))
print('E1 upper-bound training time (no early stopping): about', pilot['e1_upper_bound_hours'], 'GPU hours')
print('LiteRT FP32 and INT8 conversion works for all three architectures:', pilot['conversion_all_ok'])

In [ ]:
# 5. E1: 3 architectures x 2 crops x 3 seeds. Finished runs are skipped; safe to stop and resume.
def run_one(crop, arch, seed):
    rid = f'{crop}_{arch}_s{seed}'; rdir = f'{RUNS}/{rid}'
    if os.path.exists(f'{rdir}/done.json'): print('done already:', rid); return
    os.makedirs(rdir, exist_ok=True)
    classes = CLASSES[crop]
    tr = DATA[crop]['train_df']; se = DATA[crop]['select_df']
    keras.backend.clear_session(); keras.utils.set_random_seed(seed)
    weights = [RECIPE['class_weights'][crop][c] for c in classes]
    tds = train_ds(DATA[crop]['train_base'], len(classes), seed, weights); sel_ds = DATA[crop]['select_ds']
    y_sel = np.array([classes.index(l) for l in se.research_label])
    model, base = build_model(arch, len(classes)); log = []; t_run = time.time()
    print(f'=== {rid}: {len(tr)} training, {len(se)} selection images ===')
    model.compile(optimizer=keras.optimizers.Adam(RECIPE['head_lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(tds, epochs=RECIPE['head_max_epochs'], verbose=0,
              callbacks=[MacroF1Stopper(sel_ds, y_sel, RECIPE['patience'], log, 'head')])
    ufz = unfreeze_final_blocks(base, arch)
    model.compile(optimizer=keras.optimizers.Adam(RECIPE['finetune_lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    stopper = MacroF1Stopper(sel_ds, y_sel, RECIPE['patience'], log, 'finetune')
    stopper.best = max(r['select_macro_f1'] for r in log)                     # fine-tuning must beat the head phase
    stopper.best_w = model.get_weights()
    model.fit(tds, epochs=RECIPE['finetune_max_epochs'], verbose=0, callbacks=[stopper])
    probs = model.predict(sel_ds, verbose=0); pred = probs.argmax(1)
    model.save(f'{rdir}/best_model.keras')
    pd.DataFrame(probs, columns=[f'p_{c}' for c in classes]).assign(sample_id=list(se.sample_id), y=y_sel, pred=pred) \
      .to_csv(f'{rdir}/selection_predictions.csv', index=False)
    pd.DataFrame(log).to_csv(f'{rdir}/history.csv', index=False)
    best = max(log, key=lambda r: r['select_macro_f1'])
    info = {'run_id': rid, 'crop': crop, 'architecture': arch, 'seed': seed, 'protocol': PROTOCOL_VERSION,
            'manifest_sha256': protocol['manifest_sha256'], 'n_train': len(tr), 'n_select': len(se),
            'params_total': int(model.count_params()),
            'params_trainable_finetune': int(sum(np.prod(w.shape) for w in model.trainable_weights)),
            'unfreeze': ufz, 'best_phase': best['phase'], 'best_epoch': best['epoch'],
            'epochs_run': {'head': sum(r['phase'] == 'head' for r in log), 'finetune': sum(r['phase'] == 'finetune' for r in log)},
            'select_macro_f1': float(f1_score(y_sel, pred, average='macro')), 'select_accuracy': float(accuracy_score(y_sel, pred)),
            'select_per_class_f1': dict(zip(classes, [float(v) for v in f1_score(y_sel, pred, average=None, labels=range(len(classes)))])),
            'keras_file_bytes': os.path.getsize(f'{rdir}/best_model.keras'), 'minutes': round((time.time() - t_run) / 60, 1),
            'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'environment': env}
    json.dump(info, open(f'{rdir}/done.json', 'w'), indent=1)
    print(f'>>> {rid}: selection macro-F1 {info["select_macro_f1"]:.4f}, accuracy {info["select_accuracy"]:.4f}, {info["minutes"]} min')
    del model, base, tds; gc.collect()

DATA = {}
for crop in CROPS_TO_RUN:
    todo = [(a, s) for a in ARCHS for s in SEEDS if not os.path.exists(f'{RUNS}/{crop}_{a}_s{s}/done.json')]
    if not todo: print(crop, ': all 9 runs done'); continue
    tr = DEV[(DEV.crop == crop) & (DEV.partition == 'train')]; se = DEV[(DEV.crop == crop) & (DEV.partition == 'select')]
    DATA[crop] = {'train_df': tr, 'select_df': se, 'train_base': base_ds(tr, CLASSES[crop]),
                  'select_ds': eval_ds(base_ds(se, CLASSES[crop]), len(CLASSES[crop]))}
    for arch, seed in todo:
        run_one(crop, arch, seed)
    DATA.pop(crop); gc.collect()
print('All requested runs finished.')

In [ ]:
# 6. Summary of finished runs (selection data only), majority baseline, Figure 4.1 and results file
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt, zipfile
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{RUNS}/*/done.json'))]
S = pd.DataFrame([{k: r[k] for k in ['run_id', 'crop', 'architecture', 'seed', 'select_macro_f1', 'select_accuracy', 'params_total',
                   'params_trainable_finetune', 'best_phase', 'best_epoch', 'minutes', 'keras_file_bytes']} for r in rows])
S.to_csv(f'{OUT}/e1_runs.csv', index=False)
agg = S.groupby(['crop', 'architecture']).agg(runs=('seed', 'count'), mean_macro_f1=('select_macro_f1', 'mean'),
       sd_macro_f1=('select_macro_f1', 'std'), mean_accuracy=('select_accuracy', 'mean'), params=('params_total', 'first')).round(4)
agg.to_csv(f'{OUT}/e1_by_architecture.csv'); print(agg)
base = {}
for crop, cl in CLASSES.items():
    tr = DEV[(DEV.crop == crop) & (DEV.partition == 'train')]; se = DEV[(DEV.crop == crop) & (DEV.partition == 'select')]
    maj = tr.research_label.value_counts().idxmax(); y = (se.research_label == maj)
    base[crop] = {'majority_class': maj, 'select_accuracy': round(float(y.mean()), 4),
                  'select_macro_f1': round(float(f1_score([cl.index(l) for l in se.research_label], [cl.index(maj)] * len(se), average='macro')), 4)}
json.dump(base, open(f'{OUT}/majority_baseline.json', 'w'), indent=1); print('majority-class baseline:', base)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, crop in zip(axes, ['cassava', 'maize']):
    for r in rows:
        if r['crop'] != crop: continue
        h = pd.read_csv(f'{RUNS}/{r["run_id"]}/history.csv'); h['step'] = range(1, len(h) + 1)
        ax.plot(h.step, h.select_macro_f1, label=f'{r["architecture"]} s{r["seed"]}', lw=1)
    ax.set_title(f'{crop.capitalize()}: selection macro-F1 per epoch'); ax.set_xlabel('Epoch (head, then fine-tuning)'); ax.grid(alpha=.3)
axes[0].set_ylabel('Selection macro-F1'); axes[1].legend(fontsize=7, ncol=1, loc='lower right')
plt.tight_layout(); plt.savefig(f'{OUT}/figure_4_1_selection_curves.png', dpi=200)
res = '/content/nb03_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{OUT}/environment_nb03.json', PILOT, f'{OUT}/e1_runs.csv', f'{OUT}/e1_by_architecture.csv',
              f'{OUT}/majority_baseline.json', f'{OUT}/figure_4_1_selection_curves.png']:
        if os.path.exists(p): z.write(p, os.path.basename(p))
    for r in rows:
        for f in ['done.json', 'history.csv']: z.write(f'{RUNS}/{r["run_id"]}/{f}', f'runs/{r["run_id"]}/{f}')
from google.colab import files
files.download(res)
print(len(rows), 'of 18 runs finished')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb03_results.zip`. The copies used for the published results are in this repository under `records/nb03/`.

If Colab stops early (GPU limit), open the notebook again and **Run all**: it continues from the next unfinished run.